In [1]:
from google.colab import drive
drive.mount('/content/drive')
#

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation

/content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation


In [3]:
from pathlib import Path
import torch


In [4]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'

DATA_DIR.mkdir(parents=True, exist_ok=True)
#
print(PROJECT_ROOT)
print(DATA_DIR)

/content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation
/content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/data


In [5]:
from urllib.request import urlretrieve
from zipfile import ZipFile

DATA_URL = "https://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip"

ZIP_PATH = DATA_DIR / 'fra-eng.zip'
RAW_DATA_PATH = DATA_DIR / 'fra.txt'

# Download the dataset archive if it is not already present

if not ZIP_PATH.exists():
    urlretrieve(DATA_URL, ZIP_PATH)
    print("Dataset archive downloaded.")
else:
    print("Dataset archive already exists")

# extract the English-French text file

with ZipFile(ZIP_PATH, 'r') as zip_file:
    zip_file.extract("fra-eng/fra.txt", path=DATA_DIR)

# move the extracted file to the data directory root
EXTRACTED_PATH = DATA_DIR / 'fra-eng' / 'fra.txt'

if EXTRACTED_PATH.exists():
    EXTRACTED_PATH.replace(RAW_DATA_PATH)

# Remove the now-empty extraction folder
EXTRACTED_DIR = DATA_DIR / "fra-eng"

if EXTRACTED_DIR.exists():
    EXTRACTED_DIR.rmdir()

print("Raw dataset saved to:", RAW_DATA_PATH)
print("File exists:", RAW_DATA_PATH.exists())
print("File size:", round(RAW_DATA_PATH.stat().st_size / (1024 ** 2), 2), "MB")

Dataset archive already exists
Raw dataset saved to: /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/data/fra.txt
File exists: True
File size: 11.13 MB


In [6]:
print("Dataset exists:", RAW_DATA_PATH.exists())

Dataset exists: True


In [7]:
with open(RAW_DATA_PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print("Total lines:", len(lines))

Total lines: 167130


In [8]:
for line in lines[:10]:
    print(repr(line))

'Go.\tVa !\n'
'Hi.\tSalut !\n'
'Run!\tCours\u202f!\n'
'Run!\tCourez\u202f!\n'
'Who?\tQui ?\n'
'Wow!\tÇa alors\u202f!\n'
'Fire!\tAu feu !\n'
"Help!\tÀ l'aide\u202f!\n"
'Jump.\tSaute.\n'
'Stop!\tÇa suffit\u202f!\n'


In [9]:
for line in lines[: 10]:
    print(line)

Go.	Va !

Hi.	Salut !

Run!	Cours !

Run!	Courez !

Who?	Qui ?

Wow!	Ça alors !

Fire!	Au feu !

Help!	À l'aide !

Jump.	Saute.

Stop!	Ça suffit !



In [10]:
sample_line = "Go.\tVa !\n"

english, french = sample_line.strip().split('\t')

print("English:", english)
print("French:", french)

English: Go.
French: Va !


In [11]:
# spliting the data into english and french pairs

sentence_pairs = []

for line in lines:
    line = line.strip()
    if not line:
        continue

    parts = line.split('\t')
    if len(parts) != 2:
        continue

    english, french = parts
    sentence_pairs.append((english, french))

print("Total sentence pairs:", len(sentence_pairs))
print("First 5 pairs:")
for pair in sentence_pairs[:5]:
    print(pair)

Total sentence pairs: 167130
First 5 pairs:
('Go.', 'Va !')
('Hi.', 'Salut !')
('Run!', 'Cours\u202f!')
('Run!', 'Courez\u202f!')
('Who?', 'Qui ?')


In [12]:
# Remove leading and trailing whitespace from each sentence

cleaned_pairs = [(english.strip(), french.strip()) for english, french in sentence_pairs]

# keep pairs where both sentences are non-empty

cleaned_pairs = [
    (english, french)
    for english, french in cleaned_pairs
    if english and french
]

print("Total cleaned sentence pairs:", len(cleaned_pairs))
print("First 5 pairs:")
for pair in cleaned_pairs[:5]:
    print(pair)


Total cleaned sentence pairs: 167130
First 5 pairs:
('Go.', 'Va !')
('Hi.', 'Salut !')
('Run!', 'Cours\u202f!')
('Run!', 'Courez\u202f!')
('Who?', 'Qui ?')


In [13]:
# spliting the sentence pairs into training, test and validation pairs with ratio 80:10:10

from sklearn.model_selection import train_test_split

train_pairs, test_pairs = train_test_split(cleaned_pairs,
                                           test_size=0.1,
                                           random_state=42,
                                           shuffle=True)

train_pairs, val_pairs = train_test_split(train_pairs,
                                         test_size=1/9,
                                         random_state=42,
                                         shuffle=True)

print("Total training pairs:", len(train_pairs))
print("Total validation pairs:", len(val_pairs))
print("Total test pairs:", len(test_pairs))

Total training pairs: 133704
Total validation pairs: 16713
Total test pairs: 16713


In [14]:
# standardize text
import unicodedata, tokenizers

def normailze_text(text):
    # standardize equivalent Unicode representations.
    text = unicodedata.normalize('NFC', text)

    # Remove leading and trailing whitespace and
    # Standardize conscecutive whitespace characters.

    text = " ".join(text.split())

    return text




In [15]:
# applying normalization

train_pairs = [(normailze_text(en), normailze_text(fr)) for en, fr in train_pairs]

val_pairs = [(normailze_text(en), normailze_text(fr)) for en, fr in val_pairs]

test_pairs = [(normailze_text(en), normailze_text(fr)) for en, fr in test_pairs]

In [16]:
# inspect the results

for english, french in train_pairs[:10]:
    print("English:", repr(english))
    print("French: ", repr(french))
    print("-" * 40)


English: "You're very understanding."
French:  'Vous êtes fort compréhensifs.'
----------------------------------------
English: 'Tom had no other choice but to help Mary clean out the garage.'
French:  "Tom n'eut pas d'autre choix que d'aider Mary à nettoyer le garage."
----------------------------------------
English: 'He has a good record as a businessman.'
French:  "Il a bonne réputation comme homme d'affaire."
----------------------------------------
English: "I'm still the boss around here."
French:  "C'est encore moi le patron, ici."
----------------------------------------
English: 'Please let me try it again.'
French:  "S'il-te-plaît, laisse-moi réessayer."
----------------------------------------
English: "I can't keep it."
French:  'Je ne peux le garder.'
----------------------------------------
English: "Tom helped you, didn't he?"
French:  "Tom vous a aidées, n'est-ce pas ?"
----------------------------------------
English: 'I did it in a hurry.'
French:  "Je l'ai fait dan

In [17]:
# importing tools for tokenization
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer

In [18]:
SPECIAL_TOKEN = ["[UNK]", "[PAD]", "[BOS]", "[EOS]"]

trainer = WordLevelTrainer(special_tokens=SPECIAL_TOKEN)

print("special token", SPECIAL_TOKEN)
print("Trainer configured:", trainer is not None)

special token ['[UNK]', '[PAD]', '[BOS]', '[EOS]']
Trainer configured: True


In [19]:
# English tokenizer
english_tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))

english_tokenizer.pre_tokenizer = Whitespace()

# French tokenizer
french_tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))

french_tokenizer.pre_tokenizer = Whitespace()



In [20]:
# Extract English and French sentences from the training pairs.

english_sentence = (en for en, _ in train_pairs)

french_sentence = (fr for _, fr in train_pairs)

# # Train each tokenizer on its language's training sentences.
english_tokenizer.train_from_iterator(english_sentence, trainer=trainer)

french_tokenizer.train_from_iterator(french_sentence, trainer=trainer)

#

In [21]:

print("English vocabulary size:", english_tokenizer.get_vocab_size())
print("French vocabulary size:", french_tokenizer.get_vocab_size())

print("\nEnglish special-token IDs:")
for token in SPECIAL_TOKEN:
    print(token, ":", english_tokenizer.token_to_id(token))

print("\nFrench special-token IDs:")
for token in SPECIAL_TOKEN:
    print(token, ":", french_tokenizer.token_to_id(token))


English vocabulary size: 14493
French vocabulary size: 23539

English special-token IDs:
[UNK] : 0
[PAD] : 1
[BOS] : 2
[EOS] : 3

French special-token IDs:
[UNK] : 0
[PAD] : 1
[BOS] : 2
[EOS] : 3


In [22]:
english_sample = train_pairs[0][0]
french_sample = train_pairs[0][1]

english_encoding = english_tokenizer.encode(english_sample)
french_encoding = french_tokenizer.encode(french_sample)

print("ENGLISH")
print("Sentence:", english_sample)
print("Tokens:", english_encoding.tokens)
print("IDs:", english_encoding.ids)

print("\nFRENCH")
print("Sentence:", french_sample)
print("Tokens:", french_encoding.tokens)
print("IDs:", french_encoding.ids)

ENGLISH
Sentence: You're very understanding.
Tokens: ['You', "'", 're', 'very', 'understanding', '.']
IDs: [18, 6, 33, 64, 1892, 4]

FRENCH
Sentence: Vous êtes fort compréhensifs.
Tokens: ['Vous', 'êtes', 'fort', 'compréhensifs', '.']
IDs: [45, 69, 328, 12044, 4]


In [23]:
import numpy as np

def get_token_lengths(pairs, en_tokeninzer, fr_tokenizer):
    english_lengths = []
    french_lengths = []

    for english, french in pairs:
        english_lengths.append(len(en_tokeninzer.encode(english).ids))
        french_lengths.append(len(fr_tokenizer.encode(french).ids))

    return english_lengths, french_lengths


In [24]:
train_en_lengths, train_fr_lengths = get_token_lengths(train_pairs, english_tokenizer, french_tokenizer)

for language, lengths in [
    ("English", train_en_lengths),
    ("French", train_fr_lengths)
]:
    print(f"\n{language}")
    print("Minimum:", np.min(lengths))
    print("Median:", np.median(lengths))
    print("90th percentile:", np.percentile(lengths, 90))
    print("95th percentile:", np.percentile(lengths, 95))
    print("99th percentile:", np.percentile(lengths, 99))
    print("Maximum:", np.max(lengths))



English
Minimum: 2
Median: 7.0
90th percentile: 11.0
95th percentile: 13.0
99th percentile: 17.0
Maximum: 53

French
Minimum: 2
Median: 8.0
90th percentile: 13.0
95th percentile: 15.0
99th percentile: 20.0
Maximum: 66


In [25]:
EN_MAX_LEN = 20
FR_MAX_LEN = 22

def count_overlength(lengths, max_length):
  # Reserve two position for BOS and EOS
  total_lengths = [length + 2 for length in lengths]

  overlength = sum(
      length > max_length for length in total_lengths
  )

  print("Sentences:", len(total_lengths))
  print("Exceeding limit:", overlength)
  print(f"Percentage: {100 * overlength / len(total_lengths):.2f}%")

print("English:")
count_overlength(train_en_lengths, EN_MAX_LEN)

print("\nFrench:")
count_overlength(train_fr_lengths, FR_MAX_LEN)



English:
Sentences: 133704
Exceeding limit: 744
Percentage: 0.56%

French:
Sentences: 133704
Exceeding limit: 1014
Percentage: 0.76%


In [26]:
from tokenizers.processors import TemplateProcessing

In [27]:

english_tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[
        ("[BOS]", english_tokenizer.token_to_id("[BOS]")),
        ("[EOS]", english_tokenizer.token_to_id("[EOS]")),
    ],
)


In [28]:

french_tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[
        ("[BOS]", french_tokenizer.token_to_id("[BOS]")),
        ("[EOS]", french_tokenizer.token_to_id("[EOS]")),
    ],
)


In [29]:

english_encoding = english_tokenizer.encode(train_pairs[0][0])
french_encoding = french_tokenizer.encode(train_pairs[0][1])

print("English tokens:", english_encoding.tokens)
print("English IDs:", english_encoding.ids)

print("\nFrench tokens:", french_encoding.tokens)
print("French IDs:", french_encoding.ids)


English tokens: ['[BOS]', 'You', "'", 're', 'very', 'understanding', '.', '[EOS]']
English IDs: [2, 18, 6, 33, 64, 1892, 4, 3]

French tokens: ['[BOS]', 'Vous', 'êtes', 'fort', 'compréhensifs', '.', '[EOS]']
French IDs: [2, 45, 69, 328, 12044, 4, 3]


In [30]:
EN_MAX_LEN = 20
FR_MAX_LEN = 22

In [31]:

english_tokenizer.enable_truncation(
    max_length=EN_MAX_LEN
)

english_tokenizer.enable_padding(
    length=EN_MAX_LEN,
    pad_id=english_tokenizer.token_to_id("[PAD]"),
    pad_token="[PAD]",
)


In [32]:

french_tokenizer.enable_truncation(
    max_length=FR_MAX_LEN
)

french_tokenizer.enable_padding(
    length=FR_MAX_LEN,
    pad_id=french_tokenizer.token_to_id("[PAD]"),
    pad_token="[PAD]",
)


In [33]:

english_encoding = english_tokenizer.encode(train_pairs[0][0])
french_encoding = french_tokenizer.encode(train_pairs[0][1])

print("English token count:", len(english_encoding.ids))
print("English tokens:", english_encoding.tokens)
print("English IDs:", english_encoding.ids)

print("\nFrench token count:", len(french_encoding.ids))
print("French tokens:", french_encoding.tokens)
print("French IDs:", french_encoding.ids)


English token count: 20
English tokens: ['[BOS]', 'You', "'", 're', 'very', 'understanding', '.', '[EOS]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
English IDs: [2, 18, 6, 33, 64, 1892, 4, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

French token count: 22
French tokens: ['[BOS]', 'Vous', 'êtes', 'fort', 'compréhensifs', '.', '[EOS]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
French IDs: [2, 45, 69, 328, 12044, 4, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [34]:

# Find the longest English and French training sentences.
longest_english = max(train_pairs, key=lambda pair: len(english_tokenizer.encode(pair[0]).ids))
longest_french = max(train_pairs, key=lambda pair: len(french_tokenizer.encode(pair[1]).ids))

for language, sentence, tokenizer, max_len in [
    ("English", longest_english[0], english_tokenizer, EN_MAX_LEN),
    ("French", longest_french[1], french_tokenizer, FR_MAX_LEN),
]:
    encoding = tokenizer.encode(sentence)

    print(f"\n{language}")
    print("Original sentence:", sentence)
    print("Encoded length:", len(encoding.ids))
    print("First token:", encoding.tokens[0])
    print("Last token:", encoding.tokens[-1])
    print("EOS ID:", tokenizer.token_to_id("[EOS]"))
    print("EOS preserved:", encoding.ids[-1] == tokenizer.token_to_id("[EOS]"))



English
Original sentence: You're very understanding.
Encoded length: 20
First token: [BOS]
Last token: [PAD]
EOS ID: 3
EOS preserved: False

French
Original sentence: Vous êtes fort compréhensifs.
Encoded length: 22
First token: [BOS]
Last token: [PAD]
EOS ID: 3
EOS preserved: False


In [35]:
# function to encode english and french pairs

def encode_pairs(pairs, en_tokenizer, fr_tokenizer):
    en_sequences = []
    fr_sequences = []

    for english, french in pairs:
        en_sequence = en_tokenizer.encode(english).ids
        fr_sequence = fr_tokenizer.encode(french).ids

        en_sequences.append(en_sequence)
        fr_sequences.append(fr_sequence)

    return en_sequences, fr_sequences

In [36]:
train_en_ids, train_fr_ids = encode_pairs(train_pairs, english_tokenizer, french_tokenizer)

val_en_ids, val_fr_ids = encode_pairs(val_pairs, english_tokenizer, french_tokenizer)

test_en_ids, test_fr_ids = encode_pairs(test_pairs, english_tokenizer, french_tokenizer)

In [37]:
X_train = torch.tensor(train_en_ids, dtype=torch.long)
y_train = torch.tensor(train_fr_ids, dtype=torch.long)

X_val = torch.tensor(val_en_ids, dtype=torch.long)
y_val = torch.tensor(val_fr_ids, dtype=torch.long)

X_test = torch.tensor(test_en_ids, dtype=torch.long)
y_test = torch.tensor(test_fr_ids, dtype=torch.long)

In [38]:

print("Training:")
print("English:", X_train.shape)
print("French: ", y_train.shape)

print("\nValidation:")
print("English:", X_val.shape)
print("French: ", y_val.shape)

print("\nTest:")
print("English:", X_test.shape)
print("French: ", y_test.shape)

print("\nTensor dtype:", X_train.dtype)


Training:
English: torch.Size([133704, 20])
French:  torch.Size([133704, 22])

Validation:
English: torch.Size([16713, 20])
French:  torch.Size([16713, 22])

Test:
English: torch.Size([16713, 20])
French:  torch.Size([16713, 22])

Tensor dtype: torch.int64


In [39]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [40]:
torch.save(
    {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
    },
    PROCESSED_DIR / "translation_ids.pt",
)

In [41]:
english_tokenizer.save(
    str(PROCESSED_DIR / "english_tokenizer.json")
)

french_tokenizer.save(
    str(PROCESSED_DIR / "french_tokenizer.json")
)

In [42]:
import json

config = {
    "english_max_length": EN_MAX_LEN,
    "french_max_length": FR_MAX_LEN,
    "english_vocab_size": english_tokenizer.get_vocab_size(),
    "french_vocab_size": french_tokenizer.get_vocab_size(),
    "pad_token": "[PAD]",
    "unk_token": "[UNK]",
    "bos_token": "[BOS]",
    "eos_token": "[EOS]",
}

with open(PROCESSED_DIR / "preprocessing_config.json", "w") as file:
    json.dump(config, file, indent=4)

In [ ]:
!git status
